# train_labels 결측치 분석
- 그룹별 train_labels가 결측인 시각 분석
- 결측인 시각의 SCADA 실측 데이터와 비교하여 결측 채워넣기 or 결측 시각 삭제 (LDAPS, GFS 포함)
- _missing.csv로 저장함

# 라이브러리

In [1]:
import pandas as pd
import numpy as np

# 데이터

In [2]:
# SCADA 데이터
scada_unison_train = pd.read_csv('../../open/train/scada_unison_train.csv', encoding = 'utf-8-sig')
scada_vestas_train = pd.read_csv('../../open/train/scada_vestas_train.csv', encoding = 'utf-8-sig')

scada_unison_train['kst_dtm'] = pd.to_datetime(scada_unison_train['kst_dtm'])
scada_vestas_train['kst_dtm'] = pd.to_datetime(scada_vestas_train['kst_dtm'])

In [3]:
# Group1
ldaps_train_group1 = pd.read_csv('../../data/raw_data/ldaps_train_group1.csv', encoding = 'utf-8-sig')
ldaps_train_group1['forecast_kst_dtm'] = pd.to_datetime(ldaps_train_group1['forecast_kst_dtm'])

gfs_train_group1 = pd.read_csv('../../data/raw_data/gfs_train_group1.csv', encoding = 'utf-8-sig')
gfs_train_group1['forecast_kst_dtm'] = pd.to_datetime(gfs_train_group1['forecast_kst_dtm'])

train_labels_group1 = pd.read_csv('../../data/raw_data/train_labels_group1.csv', encoding = 'utf-8-sig')

In [4]:
# Group2
ldaps_train_group2 = pd.read_csv('../../data/raw_data/ldaps_train_group2.csv', encoding = 'utf-8-sig')
ldaps_train_group2['forecast_kst_dtm'] = pd.to_datetime(ldaps_train_group2['forecast_kst_dtm'])

gfs_train_group2 = pd.read_csv('../../data/raw_data/gfs_train_group2.csv', encoding = 'utf-8-sig')
gfs_train_group2['forecast_kst_dtm'] = pd.to_datetime(gfs_train_group2['forecast_kst_dtm'])

train_labels_group2 = pd.read_csv('../../data/raw_data/train_labels_group2.csv', encoding = 'utf-8-sig')

In [5]:
# Group3
ldaps_train_group3 = pd.read_csv('../../data/raw_data/ldaps_train_group3.csv', encoding = 'utf-8-sig')
ldaps_train_group3['forecast_kst_dtm'] = pd.to_datetime(ldaps_train_group3['forecast_kst_dtm'])

gfs_train_group3 = pd.read_csv('../../data/raw_data/gfs_train_group3.csv', encoding = 'utf-8-sig')
gfs_train_group3['forecast_kst_dtm'] = pd.to_datetime(gfs_train_group3['forecast_kst_dtm'])

train_labels_group3 = pd.read_csv('../../data/raw_data/train_labels_group3.csv', encoding = 'utf-8-sig')

# 그룹별 label 결측치 개수

In [6]:
train_labels = pd.read_csv('../../open/train/train_labels.csv', encoding = 'utf-8-sig')

In [7]:
group_cols = ["kpx_group_1", "kpx_group_2", "kpx_group_3"]

for col in group_cols:
    if col != 'kpx_group_3':
        na_rows = train_labels[train_labels[col].isna()]
        print(f"{col}: na인 행 {len(na_rows)}개")
    else:
        na_row = train_labels_group3[train_labels_group3['kpx_group_3'].isna()]
        print(f"{col}: na인 행 {len(na_row)}개")

kpx_group_1: na인 행 104개
kpx_group_2: na인 행 103개
kpx_group_3: na인 행 6개


# 그룹1 결측치 처리
- 점검으로 판단 -> 모두 제거

In [8]:
# SCADA VESTAS (1 ~ 6호기) 컬럼만 선택
group1_wtg = [f'{i:02d}' for i in range(1, 7)] # 01호기 ~ 06호기
group1_cols = ['kst_dtm'] + [
    c for c in scada_vestas_train.columns if any(f'wtg{n}_' in c for n in group1_wtg)
]

# Group1 label 결측 시각 추출
missing_times = train_labels_group1.loc[
    train_labels_group1['kpx_group_1'].isna(), 
    'kst_dtm'
]
missing_times = pd.to_datetime(missing_times)
print(f'결측 시각 개수 : {len(missing_times)}')

결측 시각 개수 : 104


In [9]:
# 해당 시각에 대응되는 행 삭제 (LDAPS)
ldaps_train_group1_missing = ldaps_train_group1[
    ~ldaps_train_group1['forecast_kst_dtm'].isin(missing_times)
].reset_index(drop = True)

print(f'삭제 후 행 수: {len(ldaps_train_group1_missing)}')

삭제 후 행 수: 419200


In [10]:
# 해당 시각에 대응되는 행 삭제 (GFS)
gfs_train_group1_missing = gfs_train_group1[
    ~gfs_train_group1['forecast_kst_dtm'].isin(missing_times)
].reset_index(drop=True)

print(f'삭제 후 행 수: {len(gfs_train_group1_missing)}')

삭제 후 행 수: 235800


In [11]:
# 해당 시각에 대응되는 행 삭제 (label)
print(f'삭제 전 행 수: {len(train_labels_group1)}')

train_labels_group1_missing = train_labels_group1[
    train_labels_group1['kpx_group_1'].notna()
].reset_index(drop=True)

print(f'삭제 후 행 수: {len(train_labels_group1_missing)}')

삭제 전 행 수: 26304
삭제 후 행 수: 26200


# 그룹2 결측치 처리
- 전부 삭제

In [12]:
group2_wtg = [f'{i:02d}' for i in range(7, 13)] # 07호기 ~ 12호기

group2_cols = ['kst_dtm'] + [
    c for c in scada_vestas_train.columns
    if any(f'wtg{n}_' in c for n in group2_wtg)
]

missing_times = train_labels_group2.loc[
    train_labels_group2['kpx_group_2'].isna(),
    'kst_dtm'
]
missing_times = pd.to_datetime(missing_times)
print(f'결측 시각 개수: {len(missing_times)}')

결측 시각 개수: 103


In [13]:
# train_labels_group2 결측 행 삭제
train_labels_group2_missing = train_labels_group2[
    train_labels_group2['kpx_group_2'].notna()
].reset_index(drop=True)
print(f'train_labels_group2_missing 행 수: {len(train_labels_group2_missing)}')

train_labels_group2_missing 행 수: 26201


In [14]:
# ldaps_train_group2에서 나머지 결측 시각 삭제
print(f'ldaps 삭제 전 행 수: {len(ldaps_train_group2)}')
ldaps_train_group2_missing = ldaps_train_group2[
    ~ldaps_train_group2['forecast_kst_dtm'].isin(missing_times)
].reset_index(drop=True)
print(f'ldaps 삭제 후 행 수: {len(ldaps_train_group2_missing)} (기대 삭제량: {len(missing_times) * 16})')

ldaps 삭제 전 행 수: 420864
ldaps 삭제 후 행 수: 419216 (기대 삭제량: 1648)


In [15]:
# gfs_train_group2에서 나머지 결측 시각 삭제
print(f'gfs 삭제 전 행 수: {len(gfs_train_group2)}')
gfs_train_group2_missing = gfs_train_group2[
    ~gfs_train_group2['forecast_kst_dtm'].isin(missing_times)
].reset_index(drop=True)
print(f'gfs 삭제 후 행 수: {len(gfs_train_group2_missing)} (기대 삭제량: {len(missing_times) * 9})')

gfs 삭제 전 행 수: 236736
gfs 삭제 후 행 수: 235809 (기대 삭제량: 927)


# 그룹3 결측치 처리
- label이 결측인 시각의 모든 SCADA 데이터가 결측 -> 모두 제거

In [16]:
group3_wtg = [f'{i:02d}' for i in range(1, 6)]  # 01~05호기
group3_cols = ['kst_dtm'] + [
    c for c in scada_unison_train.columns
    if any(f'wtg{n}_' in c for n in group3_wtg)
]

missing_times = train_labels_group3.loc[
    train_labels_group3['kpx_group_3'].isna(), 'kst_dtm'
]
missing_times = pd.to_datetime(missing_times)
print(f'결측 시각 개수: {len(missing_times)}')

결측 시각 개수: 6


In [17]:
# train_labels_group3에서 결측 행 삭제
train_labels_group3_missing = train_labels_group3[
    train_labels_group3['kpx_group_3'].notna()
].reset_index(drop=True)
print(f'train_labels_group3_missing 행 수: {len(train_labels_group3_missing)}')

train_labels_group3_missing 행 수: 17538


In [18]:
# ldaps_train_group3에서 결측 시각 삭제
print(f'ldaps 삭제 전 행 수: {len(ldaps_train_group3)}')
ldaps_train_group3_missing = ldaps_train_group3[
    ~ldaps_train_group3['forecast_kst_dtm'].isin(missing_times)
].reset_index(drop=True)
print(f'ldaps 삭제 후 행 수: {len(ldaps_train_group3_missing)} (기대 삭제량: {len(missing_times) * 16})')

ldaps 삭제 전 행 수: 420864
ldaps 삭제 후 행 수: 420768 (기대 삭제량: 96)


In [19]:
# gfs_train_group3에서 결측 시각 삭제
print(f'gfs 삭제 전 행 수: {len(gfs_train_group3)}')
gfs_train_group3_missing = gfs_train_group3[
    ~gfs_train_group3['forecast_kst_dtm'].isin(missing_times)
].reset_index(drop=True)
print(f'gfs 삭제 후 행 수: {len(gfs_train_group3_missing)} (기대 삭제량: {len(missing_times) * 9})')

gfs 삭제 전 행 수: 236736
gfs 삭제 후 행 수: 236682 (기대 삭제량: 54)


### 파일 새로 저장

ldaps_train_group1_missing.to_csv('../../data/after_missing/ldaps_train_group1_missing.csv', index = False, encoding = 'utf-8-sig')
gfs_train_group1_missing.to_csv('../../data/after_missing/gfs_train_group1_missing.csv', index = False, encoding = 'utf-8-sig')
train_labels_group1_missing.to_csv('../../data/after_missing/train_labels_group1_missing.csv', index = False, encoding = 'utf-8-sig')

ldaps_train_group2_missing.to_csv('../../data/after_missing/ldaps_train_group2_missing.csv', index = False, encoding = 'utf-8-sig')
gfs_train_group2_missing.to_csv('../../data/after_missing/gfs_train_group2_missing.csv', index = False, encoding = 'utf-8-sig')
train_labels_group2_missing.to_csv('../../data/after_missing/train_labels_group2_missing.csv', index = False, encoding = 'utf-8-sig')

ldaps_train_group3_missing.to_csv('../../data/after_missing/ldaps_train_group3_missing.csv', index = False, encoding = 'utf-8-sig')
gfs_train_group3_missing.to_csv('../../data/after_missing/gfs_train_group3_missing.csv', index = False, encoding = 'utf-8-sig')
train_labels_group3_missing.to_csv('../../data/after_missing/train_labels_group3_missing.csv', index = False, encoding = 'utf-8-sig')